In [ ]:
# Database Connection 
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.practice import Practice, q

EXPECTED = {
 "s01": [
  [
   1,
   1
  ],
  "1ba5edf59854386be920ac9fd3e4a2af3bb6369bf2a1aef78dcdf908c96d70a9"
 ],
 "s02": [
  [
   5,
   3
  ],
  "19042d389224a7f56403ad9636ed581989f6639acadf45b15461f55cd6258da7"
 ],
 "s03": [
  [
   10,
   3
  ],
  "e3e296058facb3595b46ac98fb75d445a82240fc775f3188d029eabc5a362ddb"
 ],
 "s04": [
  [
   1,
   1
  ],
  "ba4a24ee999caa9e01607eb78856b7212bcce521b0f6fd42d8aa90541c10382d"
 ],
 "s05": [
  [
   1,
   2
  ],
  "8153d8bbbc1b43b88ce7e4ea0c99a4ac07f4b19e0f62a9a1e04df3271f2ad3df"
 ]
}   # fingerprints only: the answers are not stored here
TITLES = {
 "s01": "Cleaning a messy column",
 "s02": "Dates: a date range",
 "s03": "Combining tables (JOIN)",
 "s04": "What gets lost in a join?",
 "s05": "Window: the driest summer week"
}
p = Practice(EXPECTED, TITLES)
print(f"Practice ready: {len(TITLES)} tasks. Connected as:", q("SELECT CURRENT_USER AS u").u[0])

Practice ready: 5 tasks. Connected as: intern_readonly


In [2]:
# Question 1:  Cleaning a messy column

SQL_01 = """
SELECT COUNT(*) AS days_over_350
FROM week07.london_2014
WHERE TRY_CAST(
    REPLACE([WindDirDegrees<br />], '<br />', '')
    AS int
) > 350
"""

p.sql("s01", SQL_01)

,days_over_350
0,5


✅ s01: correct!


,days_over_350
0,5


In [ ]:
# Question 2: Dates: a date range

SQL_02 = """ 
Select TRY_CONVERT(date, GMT) as day, max([Max TemperatureC]) as max_temperature, min([Min TemperatureC) as [Min TemperatureC] 
from week07.london_2014
WHERE 
    TRY_CONVERT(date, GMT) BETWEEN '2014-12-8' AND '2014-12-12' 
Group by TRY_CONVERT(date, GMT)
Order by day
"""
p.sql("s02", SQL_02)

,day,max_temperature,Min_Temperature
0,2014-12-08,7,1
1,2014-12-09,12,-1
2,2014-12-10,9,6
3,2014-12-11,10,5
4,2014-12-12,9,2


✅ s02: correct!


,day,max_temperature,Min_Temperature
0,2014-12-08,7,1
1,2014-12-09,12,-1
2,2014-12-10,9,6
3,2014-12-11,10,5
4,2014-12-12,9,2


In [45]:
# Question 3 SQL Task 3 · Combining tables (JOIN)

SQL_03 = """
Select 
TOP 10 gdp.country, ROUND([NY.GDP.MKTP.CD]/1000000000.0, 2 ) as gdp_bn_usd, ROUND([SP.DYN.LE00.IN], 1) as life_expectancy
from week08.wb_gdp_2013 gdp
inner join week08.wb_le_2013 wb on
gdp.country = wb.country
Where 
    [SP.DYN.LE00.IN] IS NOT NULL
    AND [NY.GDP.MKTP.CD] < 10000000000
order by 
    [SP.DYN.LE00.IN] desc,
    Country Asc
"""
p.sql("s03", SQL_03)

,country,gdp_bn_usd,life_expectancy
0,Faeroe Islands,2.61,81.3
1,Malta,9.64,80.7
2,Bermuda,5.57,80.6
3,Maldives,2.79,76.6
4,Antigua and Barbuda,1.20,75.8
5,Barbados,4.28,75.3
6,"Bahamas, The",8.43,75.1
7,St. Lucia,1.33,74.9
8,Montenegro,4.46,74.8
9,Seychelles,1.41,74.2


✅ s03: correct!


,country,gdp_bn_usd,life_expectancy
0,Faeroe Islands,2.61,81.3
1,Malta,9.64,80.7
2,Bermuda,5.57,80.6
3,Maldives,2.79,76.6
4,Antigua and Barbuda,1.20,75.8
5,Barbados,4.28,75.3
6,"Bahamas, The",8.43,75.1
7,St. Lucia,1.33,74.9
8,Montenegro,4.46,74.8
9,Seychelles,1.41,74.2


In [46]:
# Question 4
# SQL Task 4 · What gets lost in a join?

SQL_04 = """
Select Count(*) as rows_lost
from week08.wb_gdp_2013 gdp 
left join week08.wb_le_2013 wb
    ON gdp.country = wb.country 
where [SP.DYN.LE00.IN] IS NULL
    OR [NY.GDP.MKTP.CD] IS NULL
"""
p.sql("s04", SQL_04)

,rows_lost
0,29


✅ s04: correct!


,rows_lost
0,29


In [ ]:

SQL_05 = """
WITH summer_weather AS (
    SELECT
        TRY_CONVERT(date, GMT) AS day,
        Precipitationmm
    FROM week07.london_2014
    WHERE TRY_CONVERT(date, GMT)
        BETWEEN '2014-06-01' AND '2014-08-31'
),
rain_windows AS (
    SELECT
        day AS start_day,
        SUM(Precipitationmm) OVER (
            ORDER BY day
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS rain_mm_7d,
        COUNT(*) OVER (
            ORDER BY day
            ROWS BETWEEN CURRENT ROW AND 6 FOLLOWING
        ) AS days_in_window
    FROM summer_weather
)
SELECT TOP 1
    start_day,
    ROUND(rain_mm_7d, 2) AS rain_mm_7d
FROM rain_windows
WHERE days_in_window = 7
ORDER BY rain_mm_7d ASC, start_day ASC
"""
p.sql("s05", SQL_05)


,start_day,rain_mm_7d
0,2014-06-19,0.0


✅ s05: correct!


,start_day,rain_mm_7d
0,2014-06-19,0.0


In [49]:
p.score()

Task Type                          Topic Result
 s01  SQL        Cleaning a messy column      ✅
 s02  SQL            Dates: a date range      ✅
 s03  SQL        Combining tables (JOIN)      ✅
 s04  SQL      What gets lost in a join?      ✅
 s05  SQL Window: the driest summer week      ✅
SQL score: 5 / 5


,Task,Type,Topic,Result
0,s01,SQL,Cleaning a messy column,✅
1,s02,SQL,Dates: a date range,✅
2,s03,SQL,Combining tables (JOIN),✅
3,s04,SQL,What gets lost in a join?,✅
4,s05,SQL,Window: the driest summer week,✅
